In [ ]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import plotly.express as px
import plotly.graph_objects as go  # Add this line
from plotly.subplots import make_subplots
# Import the processing module from the same folder
sys.path.append(os.path.join("..", "scripts", "analysis"))
from processing import load_solutions, add_kwargs_as_indices, combine_solutions, read_parquet_and_convert, add_fields,apply_conservative_classification


In [ ]:
def create_envelope(s_ed, s_uc, group_by = ['configuration', 'µ', 'iteration', 'day', 'hour,', 'r_id']):
    # Copy the relevant columns from s_ed['storage']
    envelope = s_ed['storage'][group_by + ['SOE_MWh', 'envelope_up_MWh', 'envelope_down_MWh']].copy()

    # Perform the first left join
    envelope = envelope.merge(
        s_uc['storage'][[col for col in group_by if col != 'iteration'] + ['SOE_MWh', 'envelope_up_MWh', 'envelope_down_MWh']].rename(
            columns={'SOE_MWh': 'SOE_DA_MWh', 'envelope_up_MWh': 'envelope_up_DA_MWh', 'envelope_down_MWh': 'envelope_down_DA_MWh'}
        ),
        on=[col for col in group_by if col != 'iteration'],
        how='left'
    )
    
    # Perform the second left join
    envelope = envelope.merge(
        s_uc['storage_parameters'][['r_id', 'SOE_max_MWh', 'initial_energy_proportion']].drop_duplicates(),
        on='r_id',
        how='left'
    )
    
    # Calculate initial state of energy (SOE) based on maximum SOE and initial energy proportion
    envelope['SOE_0_MWh'] = envelope['SOE_max_MWh'] * envelope['initial_energy_proportion']
    envelope['SOC'] = envelope['SOE_MWh'] / envelope['SOE_max_MWh'] 
    # Group by day, configuration, and resource ID, and get the last entry for each group

    return envelope

In [ ]:
ss = [
    # {'solution_folder': f"RTS-GMLC_v3.1.1s", 'VLGEN': 30, 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v4.1.1s", 'VLGEN': 30, 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v28.0s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v28.1s", 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v_s1.2s", 'VLGEN': 1e3, 'model_type' : 'stochastic'}
    {'solution_folder': f"RTS-GMLC_v32.3s", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v32.1s", 'model_type' : 'e-reserve'},
]
days = [131]
s_uc = []
s_ed = []
gcd_KPI_adequacy = []
gcdi_KPI_adequacy = []
solution_keys = ['storage_parameters','storage', 'dual_variables', 'reserve', 'energy_reserve']
for sol in ss:
    # ρ = sol['ρ']
    s = sol['solution_folder']
    # s_uc_name = 's_uc' if sol['model_type'] == 'stochastic' else 's_uc'
    # s_ed_name = 's_sed'
    s_uc_ = load_solutions("s_uc", os.path.join("..", "output", s), days, solution_keys = solution_keys,  model_type = sol['model_type'], solution_id = s)
    if sol['model_type'] != 'stochastic':
        s_ed_ = load_solutions("s_ed", os.path.join("..", "output", s), days, solution_keys = solution_keys, model_type = sol['model_type'], solution_id = s)
    else:
        s_ed_ = load_solutions("s_suc", os.path.join("..", "output", s), days, solution_keys = solution_keys, model_type = sol['model_type'], solution_id = s)
    s_uc.append(s_uc_)
    s_ed.append(s_ed_)

    # gcd_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcd_KPI_adequacy.parquet"))
    # gcd_KPI_adequacy_ = add_fields(gcd_KPI_adequacy_, model_type = sol['model_type'], ρ=ρ, solution_id = s) 

    # gcdi_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcdi_KPI_adequacy.parquet"))
    # gcdi_KPI_adequacy_ = add_fields(gcdi_KPI_adequacy_, model_type = sol['model_type'], ρ=ρ, solution_id = s)

    # gcd_KPI_adequacy.append(gcd_KPI_adequacy_)
    # gcdi_KPI_adequacy.append(gcdi_KPI_adequacy_)

s_uc = combine_solutions(s_uc)
s_ed = combine_solutions(s_ed)
# gcd_KPI_adequacy = pd.concat(gcd_KPI_adequacy)
# gcdi_KPI_adequacy = pd.concat(gcdi_KPI_adequacy)

for k,v in s_uc.items():
    if 'µ' in v.columns:
        s_uc[k] = apply_conservative_classification(s_uc[k])
for k,v in s_ed.items():
    if 'µ' in v.columns:
        s_ed[k]= apply_conservative_classification(s_ed[k])

# if 'µ' in gcdi_KPI_adequacy.columns: 
# #         # out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: (x[0] !='envelope') + (x[0] =='envelope')*(x[1]<1), axis = 1)
#     gcdi_KPI_adequacy['model_type'] = gcdi_KPI_adequacy.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)
#     gcd_KPI_adequacy['model_type'] = gcd_KPI_adequacy.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)


storage_ids = range(101,111)
for key in s_uc.keys():
    s_uc[key] = s_uc[key][s_uc[key].r_id.isin(storage_ids)]
for key in s_ed.keys():    
    s_ed[key] = s_ed[key][s_ed[key].r_id.isin(storage_ids)] 



In [ ]:
da_storage_reserve = s_uc['reserve'][s_uc['reserve'].resource == 'battery'][['hour','day','model_type','r_id','reserve_up_MW', 'reserve_down_MW']].copy()
da_storage_reserve.dropna(inplace = True) # we drop energy reserves with NaN values
da_storage_e_reserve = s_uc['energy_reserve'][s_uc['energy_reserve'].resource == 'battery'][['hour','hour_i','day','model_type','r_id','energy_reserve_up_MW', 'energy_reserve_down_MW']].copy()


In [ ]:
tuples = [(model_type, day, r_id, h_i, h) for day in da_storage_reserve.day.unique() for model_type in da_storage_reserve.model_type.unique() for r_id in da_storage_reserve.r_id.unique() for h in da_storage_reserve.hour.unique() for h_i in da_storage_reserve.hour.unique() if h_i <= h]
da_storage_cum_reserve = pd.DataFrame(tuples, columns=['model_type', 'day', 'r_id', 'hour_i', 'hour'])
da_storage_cum_reserve = da_storage_cum_reserve.merge(da_storage_reserve, left_on = ['model_type', 'day', 'r_id', 'hour'], right_on = ['model_type', 'day', 'r_id', 'hour'], how = 'left')
da_storage_cum_reserve.set_index(['model_type', 'day', 'r_id', 'hour_i', 'hour'], inplace=True)
da_storage_cum_reserve = da_storage_cum_reserve.groupby(['model_type', 'day', 'r_id', 'hour_i']).cumsum().reset_index()
da_storage_cum_reserve = da_storage_cum_reserve.rename(columns={
    'reserve_up_MW': 'cum_reserve_up_MW',
    'reserve_down_MW': 'cum_reserve_down_MW'
})


In [ ]:
group_storage = True
group_energy_envelopes = False

In [ ]:
# da_storage_e_reserve = da_storage_e_reserve.groupby(['hour','day','model_type','r_id']).agg({'energy_reserve_up_MW':'max', 'energy_reserve_down_MW':'min'}).reset_index()
# Rename columns to match da_storage_reserve
da_storage_e_reserve_ = da_storage_e_reserve.rename(columns={
    'energy_reserve_up_MW': 'reserve_up_MW',
    'energy_reserve_down_MW': 'reserve_down_MW'
})
# da_storage_e_reserve_ = da_storage_e_reserve_.groupby(['hour','day','model_type','r_id']).agg({'reserve_up_MW':'max', 'reserve_down_MW':'max'}).reset_index()
da_storage_e_reserve_ = da_storage_e_reserve_[da_storage_e_reserve_.hour_i == da_storage_e_reserve_.hour].reset_index()
da_storage_reserve= pd.concat([da_storage_reserve, da_storage_e_reserve_], ignore_index=True)

In [ ]:
da_storage_dispatch = s_uc['storage'][['hour','day','model_type', 'hour_i','r_id','charge_MW', 'discharge_MW']].copy()
da_storage_dispatch = da_storage_dispatch.groupby(['hour','day','model_type','r_id']).agg({'charge_MW':'max', 'discharge_MW':'min'}).reset_index()
da_storage_dispatch['net_charge_MW'] = da_storage_dispatch['charge_MW'] - da_storage_dispatch['discharge_MW']
# Merge with da_storage_reserve
da_storage_dispatch = da_storage_dispatch.merge(da_storage_reserve, on=['hour','day','model_type','r_id'], how='outer', suffixes=('_orig', '_e_reserve'))
if group_storage:
    da_storage_dispatch = da_storage_dispatch.groupby(['hour','day','model_type']).sum().reset_index()
# da_storage_dispatch = da_storage_dispatch.groupby(['hour','day', 'model_type']).sum().reset_index()

In [ ]:
day_ = 131
# group_by = ['hour','ρ','model_type']
# envelope = create_envelope(s_ed, s_uc, group_by = group_by)

da_SOE =  s_uc['storage'][['hour','day','model_type', 'hour_i','r_id','SOE_MWh', 'envelope_up_MWh', 'envelope_down_MWh']].copy()
da_SOE = da_SOE.merge(
        s_uc['storage_parameters'][['r_id', 'SOE_max_MWh']].drop_duplicates(),
        on=['r_id'],
        how='left',
    )
da_SOE.loc[da_SOE.hour_i.isna(),'hour_i'] = da_SOE.loc[da_SOE.hour_i.isna(),'hour']


if group_storage:
    # if group_energy_envelopes:
    #     groupby_ = ['hour','day','model_type']
    # else:
    #     groupby_ = ['hour','hour_i','day', 'model_type']
    da_SOE = da_SOE.groupby(['hour','hour_i','day', 'model_type']).sum().reset_index()


if group_energy_envelopes:
    if group_storage:
        groupby_ = ['hour','day','model_type']
    else:
        groupby_ = ['hour','day', 'model_type', 'r_id']

    da_SOE = da_SOE.groupby(groupby_).agg(
        {'SOE_max_MWh':'max','SOE_MWh':'max','envelope_up_MWh': 'max', 'envelope_down_MWh': 'min'}
        ).reset_index()
    # da_SOE = da_SOE.groupby(['hour','day','r_id','model_type']).max().reset_index()
    # da_SOE.dropna(inplace = True)
    
# da_SOE = da_SOE.reset_index()
# if group_storage:
#     if group_energy_envelopes:
#         groupby_ = ['hour','day','model_type']
#     else:
#         groupby_ = ['hour','hour_i','day', 'model_type']
#     da_SOE = da_SOE.groupby(groupby_).sum()

da_SOE['envelope_down'] = (da_SOE['envelope_down_MWh']) / da_SOE['SOE_max_MWh']
da_SOE['envelope_up'] = (da_SOE['envelope_up_MWh']) / da_SOE['SOE_max_MWh']
da_SOE['SOC'] = da_SOE['SOE_MWh'] / da_SOE['SOE_max_MWh']
if group_storage:
    da_SOE['r_id'] = 'all'
# da_SOE = da_SOE.reset_index()

In [ ]:
s_uc['storage']

In [ ]:
time_zone = 0
fig = px.line(
    da_SOE[(da_SOE.day == day_) & (da_SOE.hour-time_zone == (da_SOE.hour_i) if not group_energy_envelopes else True)],
    x='hour',
    y=['envelope_down', 'envelope_up'],
    # color_discrete_map={
    #     'envelope_down_MWh': 'blue',
    #     'envelope_up_MWh': 'purple'
    # },
    # line_dash='variable',
    color='model_type',
    # facet_row = 'model_type',
    facet_col = 'r_id',
    # line_group = 'ρ',
    # line_dash ='ρ',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve"]},  # Explicitly define the order
    markers=True,  # Adds markers to the lines
    labels = {'value' :'SOC'},
)
# fig.update_yaxe(title_text="SOE [MWh]")
# Increase figure size
legend_attr = dict(
    x=0.5,
    y=-0.25,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)
fig.update_layout(legend=legend_attr, height=500, width=700)
fig.show()

In [ ]:
da_SOE_time_zone = []
for time_zone in da_SOE.hour.unique():
   
    aux = da_SOE[(da_SOE.day == day_) & (da_SOE.model_type == 'e-reserve') & (da_SOE.hour-time_zone == (da_SOE.hour_i) if not group_energy_envelopes else True)].copy()

    aux = da_SOE[(da_SOE.day == day_) & (da_SOE.model_type == 'e-reserve') & (da_SOE.hour-time_zone == (da_SOE.hour_i) if not group_energy_envelopes else True)].copy()
    
    
    aux['time_zone']  = time_zone
    da_SOE_time_zone.append(aux)

da_SOE_time_zone = pd.concat(da_SOE_time_zone, ignore_index=True)


In [ ]:
fig = px.line(
     da_SOE[(da_SOE.day == day_)& (da_SOE.model_type == 'e-reserve') ] ,
    x='hour',
    y=['envelope_down', 'envelope_up'],
    color='hour_i',
    facet_col = 'r_id',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve"]},  # Explicitly define the order
    markers=True,  # Adds markers to the lines
    labels = {'value' :'SOC'},
)
legend_attr = dict(
    x=0.5,
    y=-0.45,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)
fig.update_layout(legend=legend_attr, height=500, width=700)
fig.show()

In [ ]:
fig = px.line(
    da_SOE_time_zone,
    x='hour',
    y=['envelope_down', 'envelope_up'],
    color='time_zone',
    facet_col = 'r_id',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve"]},  # Explicitly define the order
    markers=True  # Adds markers to the lines
)
fig.show()
#

In [ ]:

fig = px.line(
    da_SOE[(da_SOE.day == day_)],
    x='hour',
    y=['SOE_MWh','envelope_down_MWh', 'envelope_up_MWh','SOE_max_MWh'],
    # color_discrete_map={
    #     'envelope_down_MWh': 'blue',
    #     'envelope_up_MWh': 'purple'
    # },
    # line_dash='variable',
    color='model_type',
    # facet_row = 'model_type',
    facet_col = 'r_id',
    # line_group = 'ρ',
    # line_dash ='ρ',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve"]},  # Explicitly define the order
    markers=True  # Adds markers to the lines
)
# fig.update_yaxe(title_text="SOE [MWh]")
# Increase figure size
fig.show()

In [ ]:

fig = px.line(
    da_SOE[(da_SOE.day == day_)&(da_SOE.hour == da_SOE.hour_i)],
    x='hour',
    y=['SOE_MWh','envelope_down_MWh', 'envelope_up_MWh','SOE_max_MWh'],
    # color_discrete_map={
    #     'envelope_down_MWh': 'blue',
    #     'envelope_up_MWh': 'purple'
    # },
    # line_dash='variable',
    color='model_type',
    # facet_row = 'model_type',
    facet_col = 'r_id',
    # line_group = 'ρ',
    # line_dash ='ρ',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve"]},  # Explicitly define the order
    markers=True  # Adds markers to the lines
)
# fig.update_yaxe(title_text="SOE [MWh]")
# Increase figure size
fig.show()
#

In [ ]:
da_storage_dispatch_stacked = da_storage_dispatch.melt(
    id_vars=['hour', 'day', 'model_type', 'r_id'],  # columns to keep as identifiers
    value_vars=['net_charge_MW', 'reserve_up_MW', 'reserve_down_MW'],  # columns to stack
    var_name='metric',  # name for the new column containing the variable names
    value_name='value'     # name for the new column containing the values
)

In [ ]:
fig = px.bar(
    da_storage_dispatch_stacked[da_storage_dispatch_stacked.day == day_],
    x='hour',
    y='value',
    facet_row='model_type',
    # style='metric',
    color = 'metric',
    facet_col = 'r_id',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve"]},  # Explicitly define the order
    # markers=True  # Adds markers to the lines
)

fig.show()
#
#

In [ ]:
envelope_dual = s_uc['dual_variables'][[
    'hour','hour_i','day','r_id','model_type',
    'dual_SOE_up_max_MU_MW', 'dual_SOE_down_max_MU_MW', 'dual_SOE_up_min_MU_MW', 'dual_SOE_down_min_MU_MW',
    'dual_ESOE_up_max_MU_MW', 'dual_ESOE_down_max_MU_MW', 'dual_ESOE_up_min_MU_MW', 'dual_ESOE_down_min_MU_MW',
    ]].copy()
storage_list =[101,102,103,104,105,106,107,108,109,110]
envelope_dual = envelope_dual[envelope_dual['r_id'].isin(storage_list)]
# envelope_dual['hour'] = envelope_dual['hour'] - (envelope_dual['day']-1)*24

In [ ]:
fig = px.line(
    envelope_dual[envelope_dual.day == day_],
    x='hour',
    y=['dual_SOE_up_max_MU_MW', 'dual_ESOE_up_max_MU_MW'],
    # color_discrete_map={
    #     'dual_SOE_up_max_MU_MW': 'blue',
    #     'dual_SOE_down_min_MU_MW': 'purple'
    # },
    color='model_type',
    facet_col = 'r_id',
    # facet_row='model_type',
    # color = 'day',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve"]},  # Explicitly define the order
    markers=True  # Adds markers to the lines
)

fig.show()

In [ ]:
fig = px.line(
    envelope_dual[envelope_dual.day == day_],
    x='hour',
    y=['dual_SOE_down_min_MU_MW', 'dual_ESOE_down_min_MU_MW'],
    # color_discrete_map={
    #     'dual_SOE_up_max_MU_MW': 'blue',
    #     'dual_SOE_down_min_MU_MW': 'purple'
    # },
    color='model_type',
    facet_col = 'r_id',
    facet_row='model_type',
    # color = 'day',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve"]},  # Explicitly define the order
    markers=True  # Adds markers to the lines
)
fig.show()